# 04. Calibration Observers: MinMax, Moving Average, Percentile, MSE, KL, and ACIQ

- **Category:** 14_quantisation / 01_foundations_formats_and_observers
- **Lineage & Citations:** Jacob et al. (CVPR 2018, *Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference*); NVIDIA (2017, *TensorRT 3.0 8-Bit Inference with Entropy Calibration*); Banner et al. (NeurIPS 2019, *Post-Training 4-bit Quantization of Convolutional Networks for Rapid-Deployment via ACIQ*).
- **Purpose:** Formulate post-training quantization calibration observers to compute dynamic clipping thresholds $[-\alpha, \alpha]$ balancing clipping noise against rounding variance across MinMax, Moving Average, Percentile, MSE grid search, TensorRT KL divergence, and ACIQ analytical closed-form estimators.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. Quantile and Information Theoretic Quantization](03_quantile_and_information_theoretic_nf4_sf4_fp4_lloyd_max_and_dynamic_tree.ipynb) $\to$ `🔗 Downstream Syntheses:` [01. Torch Export PT2E Quantization Flow](../02_pt2e_torchao_and_bitpacking/01_pt2e_export_quantization_workflow.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use raw MinMax observers on transformer attention activations with heavy outlier channels (outliers inflate $\alpha$, degrading precision across $99.9\%$ of tokens); do NOT apply Gaussian ACIQ to heavily skewed non-Gaussian activation distributions.

## Mathematical Foundations & Calibration Formulations

1. **MinMax & Moving Average Observers**:
   $$\alpha_{\text{MinMax}} = \max_i |x_i|, \quad \alpha_t = (1 - c) \cdot \alpha_{t-1} + c \cdot \max_i |x_{t, i}|$$
2. **Percentile Observer**:
   $$\alpha_p = \inf \{ \alpha : F_{|X|}(\alpha) \ge p \}, \quad p = 0.95 \text{ or } 0.999$$
3. **MSE Optimization Observer**:
   $$\alpha^* = \arg\min_{\alpha} \mathbb{E}[(X - Q(X; \alpha))^2] \quad \text{via grid search over } [0.55 \alpha_{\max}, \alpha_{\max}]$$
4. **TensorRT Entropy (KL Divergence) Calibration**:
   $$D_{\text{KL}}(P \parallel Q) = \sum_{i} P(i) \log \left( \frac{P(i)}{Q(i)} \right) \implies T^* = \arg\min_T D_{\text{KL}}(P_T \parallel Q_T)$$
5. **Analytical Clipping for Integer Quantization (ACIQ)**:
   $$\alpha^* = \beta \cdot \sigma \quad \text{where } \beta_{4\text{-bit}}^{\text{Gauss}} = 2.8279, \quad \beta_{4\text{-bit}}^{\text{Laplace}} = 3.89.$$

In [ ]:
import sys, time, math
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/14_quantisation/01_foundations_formats_and_observers/04_calibration_observers_minmax_percentile_mse_kl_and_aciq.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def linear_anchor(x, weight, bias):
    return x @ weight + bias

def quantize_dequantize_symmetric(x, alpha, qmax=7.0):
    if alpha == 0.0: return torch.zeros_like(x)
    scale = alpha / qmax
    return torch.clamp(torch.round(x / scale), -qmax, qmax) * scale

def minmax_observer(x, qmax=7.0):
    min_val, max_val, amax = float(torch.min(x)), float(torch.max(x)), float(torch.max(torch.abs(x)))
    return quantize_dequantize_symmetric(x, amax, qmax), torch.tensor([min_val, max_val, amax])

def moving_average_minmax_observer(batches, momentum=0.1):
    rmin, rmax = float(torch.min(batches[0])), float(torch.max(batches[0]))
    for b in range(1, batches.shape[0]):
        rmin = (1.0 - momentum) * rmin + momentum * float(torch.min(batches[b]))
        rmax = (1.0 - momentum) * rmax + momentum * float(torch.max(batches[b]))
    return torch.tensor([rmin, rmax])

def percentile_observer(x, percentile=0.95, qmax=7.0):
    sorted_abs, _ = torch.sort(torch.abs(x.flatten()))
    alpha = float(sorted_abs[int(percentile * float(len(sorted_abs) - 1))])
    return quantize_dequantize_symmetric(x, alpha, qmax), torch.tensor([alpha])

def mse_observer(x, num_candidates=10, qmax=7.0):
    amax = float(torch.max(torch.abs(x)))
    best_mse, best_alpha = float('inf'), amax
    for k in range(num_candidates):
        cand = amax * (0.55 + 0.05 * float(k))
        cand_deq = quantize_dequantize_symmetric(x, cand, qmax)
        mse = float(torch.mean((x - cand_deq) ** 2))
        if mse < best_mse: best_mse, best_alpha = mse, cand
    return quantize_dequantize_symmetric(x, best_alpha, qmax), torch.tensor([best_alpha])

def kl_divergence_observer(x, num_bins=64, num_quant_bins=8, qmax=7.0):
    x_abs, amax = torch.abs(x.flatten()), float(torch.max(torch.abs(x)))
    hist = torch.bincount(torch.clamp((x_abs / (amax / num_bins)).long(), 0, num_bins - 1), minlength=num_bins).float()
    best_kl, best_t = float('inf'), num_bins
    for T in range(num_quant_bins, num_bins + 1, 8):
        P = hist[:T].clone()
        P[-1] += torch.sum(hist[T:])
        bsz, Q = T // num_quant_bins, torch.zeros_like(P)
        for k in range(num_quant_bins):
            chunk = P[k * bsz:(k + 1) * bsz]
            nz = float(torch.sum((chunk > 0).float()))
            if nz > 0: Q[k * bsz:(k + 1) * bsz] = torch.where(chunk > 0, torch.sum(chunk) / nz, 0.0)
        P_s, Q_s = P + 1e-7, Q + 1e-7
        p_n, q_n = P_s / torch.sum(P_s), Q_s / torch.sum(Q_s)
        kl = float(torch.sum(p_n * torch.log(p_n / q_n)))
        if kl < best_kl: best_kl, best_t = kl, T
    best_alpha = (float(best_t) / float(num_bins)) * amax
    return quantize_dequantize_symmetric(x, best_alpha, qmax), torch.tensor([best_alpha])

def aciq_gaussian_observer(x, qmax=7.0):
    mu = float(torch.mean(x))
    alpha = 2.8279 * float(torch.sqrt(torch.mean((x - mu) ** 2)))
    return quantize_dequantize_symmetric(x, alpha, qmax), torch.tensor([alpha])

def aciq_laplace_observer(x, qmax=7.0):
    mu = float(torch.mean(x))
    alpha = 3.89 * float(torch.mean(torch.abs(x - mu)))
    return quantize_dequantize_symmetric(x, alpha, qmax), torch.tensor([alpha])

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert torch.max(torch.abs(y_anchor - weights["test_output"])) < 1e-6

# 2. MinMax Observer Parity
minmax_deq, minmax_s = minmax_observer(weights["raw_act"])
assert torch.max(torch.abs(minmax_deq - weights["minmax_dequant"])) < 1e-5
assert torch.max(torch.abs(minmax_s - weights["minmax_stats"])) < 1e-5

# 3. Moving Average MinMax Parity
mov_s = moving_average_minmax_observer(weights["raw_batches"], momentum=0.1)
assert torch.max(torch.abs(mov_s - weights["moving_avg_stats"])) < 1e-5

# 4. Percentile Observer Parity
perc_deq, perc_a = percentile_observer(weights["raw_act"], percentile=0.95)
assert torch.max(torch.abs(perc_deq - weights["percentile_dequant"])) < 1e-5
assert torch.max(torch.abs(perc_a - weights["percentile_alpha"])) < 1e-5

# 5. MSE Observer Parity
mse_deq, mse_a = mse_observer(weights["raw_act"], num_candidates=10)
assert torch.max(torch.abs(mse_deq - weights["mse_dequant"])) < 1e-5
assert torch.max(torch.abs(mse_a - weights["mse_alpha"])) < 1e-5

# 6. KL Divergence Observer Parity
kl_deq, kl_a = kl_divergence_observer(weights["raw_act"], num_bins=64, num_quant_bins=8)
assert torch.max(torch.abs(kl_deq - weights["kl_dequant"])) < 1e-5
assert torch.max(torch.abs(kl_a - weights["kl_alpha"])) < 1e-5

# 7. ACIQ Gaussian Parity
ag_deq, ag_a = aciq_gaussian_observer(weights["raw_act"])
assert torch.max(torch.abs(ag_deq - weights["aciq_gauss_dequant"])) < 1e-5
assert torch.max(torch.abs(ag_a - weights["aciq_gauss_alpha"])) < 1e-5

# 8. ACIQ Laplace Parity
al_deq, al_a = aciq_laplace_observer(weights["raw_laplace"])
assert torch.max(torch.abs(al_deq - weights["aciq_laplace_dequant"])) < 1e-5
assert torch.max(torch.abs(al_a - weights["aciq_laplace_alpha"])) < 1e-5
print("All 8 calibration observer parity assertions verified!")

In [ ]:
# Microbenchmark: ACIQ (Closed-Form) vs MSE (Grid Search) vs KL (Histogram)
x_bench = weights["raw_act"].repeat(100)
t0 = time.perf_counter()
for _ in range(100): _ = aciq_gaussian_observer(x_bench)
t_aciq = (time.perf_counter() - t0) / 100 * 1e6

t0 = time.perf_counter()
for _ in range(100): _ = mse_observer(x_bench, num_candidates=10)
t_mse = (time.perf_counter() - t0) / 100 * 1e6

t0 = time.perf_counter()
for _ in range(100): _ = kl_divergence_observer(x_bench, num_bins=64, num_quant_bins=8)
t_kl = (time.perf_counter() - t0) / 100 * 1e6

print(f"ACIQ Observer latency:   {t_aciq:.2f} us (O(N) single pass)")
print(f"MSE Observer latency:    {t_mse:.2f} us ({t_mse / t_aciq:.1f}x slower)")
print(f"KL Observer latency:     {t_kl:.2f} us ({t_kl / t_aciq:.1f}x slower)")

In [ ]:
# Calibration Quality: MinMax vs MSE vs ACIQ under Outlier Contamination
torch.manual_seed(42)
clean_act = torch.randn(2048)
# Inject 0.2% extreme outliers (e.g. attention spike tokens)
outliers = clean_act.clone()
outliers[torch.randint(0, 2048, (4,))] = torch.tensor([15.0, -18.0, 14.5, -16.0])

# 1. Raw MinMax (Vulnerable to single outlier)
q_minmax, s_minmax = minmax_observer(outliers)
mse_minmax = float(torch.mean((clean_act - q_minmax) ** 2))
sqnr_minmax = 10.0 * math.log10(float(torch.mean(clean_act ** 2)) / mse_minmax)

# 2. MSE Grid Search (Clips outliers)
q_mse, s_mse = mse_observer(outliers, num_candidates=10)
mse_opt = float(torch.mean((clean_act - q_mse) ** 2))
sqnr_mse = 10.0 * math.log10(float(torch.mean(clean_act ** 2)) / mse_opt)

# 3. ACIQ Gaussian (Closed-form robust std)
q_aciq, s_aciq = aciq_gaussian_observer(outliers)
mse_aciq = float(torch.mean((clean_act - q_aciq) ** 2))
sqnr_aciq = 10.0 * math.log10(float(torch.mean(clean_act ** 2)) / mse_aciq)

print(f"MinMax SQNR with outliers: {sqnr_minmax:.2f} dB (Scale blown out to {s_minmax[2]:.1f})")
print(f"MSE Observer SQNR:         {sqnr_mse:.2f} dB (Scale clipped to {s_mse[0]:.2f})")
print(f"ACIQ Observer SQNR:        {sqnr_aciq:.2f} dB (Scale analytical {s_aciq[0]:.2f})")
gain = sqnr_aciq - sqnr_minmax
print(f"ACIQ Clipping SQNR Advantage: +{gain:.2f} dB over raw MinMax!")
assert gain > 10.0, "Outlier-resistant observers must beat MinMax by >10 dB SQNR!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Clipping vs Rounding Balance**: Dynamic threshold calibration minimizes joint distortion $E = E_{\text{clip}} + E_{\text{round}}$, preventing outlier inflation from coarsening inner quantization bins.
2. **ACIQ Analytical Power**: Banner et al.'s closed-form $\alpha = \beta \cdot \sigma$ yields optimal clipping thresholds in single-pass $\mathcal{O}(N)$ time, running $10\times$ faster than iterative search.
3. **Entropy Calibration Robustness**: TensorRT KL divergence minimizes information loss across distribution histograms, yielding high SQNR on non-standard multimodal activation patterns.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_calibration_observers_minmax_percentile_mse_kl_and_aciq.md](../../../markdown/14_quantisation/01_foundations_formats_and_observers/04_calibration_observers_minmax_percentile_mse_kl_and_aciq.md)
- **PyTorch 2.x Module:** [04_calibration_observers_minmax_percentile_mse_kl_and_aciq.py](../../../pytorch2/14_quantisation/01_foundations_formats_and_observers/04_calibration_observers_minmax_percentile_mse_kl_and_aciq.py)
- **Functional JAX Module:** [04_calibration_observers_minmax_percentile_mse_kl_and_aciq.py](../../../jax/14_quantisation/01_foundations_formats_and_observers/04_calibration_observers_minmax_percentile_mse_kl_and_aciq.py)
- **Native Rust Candle Track:** [04_calibration_observers_minmax_percentile_mse_kl_and_aciq.rs](../../../candle/14_quantisation/01_foundations_formats_and_observers/04_calibration_observers_minmax_percentile_mse_kl_and_aciq.rs)
- **Hardware Kernel (CPU / WGSL):** [04_calibration_observers_minmax_percentile_mse_kl_and_aciq.rs](../../../kernels/14_quantisation/01_foundations_formats_and_observers/04_calibration_observers_minmax_percentile_mse_kl_and_aciq.rs)
- **Golden SafeTensors:** [04_calibration_observers_minmax_percentile_mse_kl_and_aciq.safetensors](../../../weights/14_quantisation/01_foundations_formats_and_observers/04_calibration_observers_minmax_percentile_mse_kl_and_aciq.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. Quantile and Information Theoretic Quantization](03_quantile_and_information_theoretic_nf4_sf4_fp4_lloyd_max_and_dynamic_tree.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 01. Torch Export PT2E Quantization Flow](../02_pt2e_torchao_and_bitpacking/01_pt2e_export_quantization_workflow.ipynb) &rarr;